In [ ]:
!pip install pdfminer.six

In [1]:
# Imports
import fitz
import csv
import cv2
import numpy as np
import os
from pdfminer.high_level import extract_pages
from pdfminer.layout import (
    LTTextBox, LTTextLine, LTFigure, LTImage, LTLine
)

print('Imports done')

Imports done


In [2]:
# Config
PDF   = "AMER_A&D_Defense_General Dynamics_Q1 (Jan-Mar).pdf"
DPI   = 150
SCALE = DPI / 72.0

COLORS = {
    'text':   (255, 80,  80),
    'figure': (50,  180, 50),
    'image':  (50,  180, 50),
    'line':   (180, 180, 180),
}

In [3]:
# Extract elements from PDF
def extract_elements(pdf_path):
    elements = []
    
    for page_num, page_layout in enumerate(extract_pages(pdf_path), start=1):
        page_h = page_layout.height

        for element in page_layout:
            bbox = element.bbox

            if isinstance(element, LTTextBox):
                for line in element:
                    if not isinstance(line, LTTextLine):
                        continue
                    text = line.get_text().strip()
                    if not text:
                        continue
                    elements.append({
                        'kind': 'text',
                        'page_num': page_num,
                        'bbox_pdf': line.bbox,
                        'page_h': page_h,
                        'content': text
                    })

            elif isinstance(element, LTImage):
                elements.append({
                    'kind': 'image',
                    'bbox_pdf': bbox,
                    'page_num': page_num,
                    'page_h': page_h,
                    'content': '[IMAGE]'
                })

            elif isinstance(element, LTFigure):
                elements.append({
                    'kind': 'figure',
                    'bbox_pdf': bbox,
                    'page_num': page_num,
                    'page_h': page_h,
                    'content': '[FIGURE]'
                })

            elif isinstance(element, LTLine):
                elements.append({
                    'kind': 'line',
                    'bbox_pdf': bbox,
                    'page_num': page_num,
                    'page_h': page_h,
                    'content': '[LINE]'
                })

    return elements

In [4]:
# Assign reading order (2-column aware)
def assign_reading_order(elements):
    from collections import defaultdict

    pages = defaultdict(list)
    for e in elements:
        pages[e['page_num']].append(e)

    counter = 1

    for page_num in sorted(pages.keys()):
        page_elements = pages[page_num]

        x_centers = [(e['bbox_pdf'][0] + e['bbox_pdf'][2]) / 2 for e in page_elements]
        page_mid  = (min(x_centers) + max(x_centers)) / 2

        for e in page_elements:
            x_center = (e['bbox_pdf'][0] + e['bbox_pdf'][2]) / 2
            e['column'] = 0 if x_center < page_mid else 1

        page_elements.sort(key=lambda e: (e['column'], -e['bbox_pdf'][3]))

        for e in page_elements:
            e['reading_order'] = counter
            counter += 1

    return elements

In [5]:
# Convert PDF bbox → image bbox
def attach_image_bboxes(elements, scale):
    for e in elements:
        x0, y0, x1, y1 = e['bbox_pdf']
        ph = e['page_h']

        e['bbox_img'] = (
            int(x0 * scale),
            int((ph - y1) * scale),
            int(x1 * scale),
            int((ph - y0) * scale)
        )

    return elements

In [6]:
# Render + annotate pages
def render_and_annotate(pdf_path, elements, scale, out_folder='annotated_pages'):
    os.makedirs(out_folder, exist_ok=True)

    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        img = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        page_elements = [e for e in elements if e['page_num'] == page_num]

        for e in page_elements:
            ix0, iy0, ix1, iy1 = e['bbox_img']
            color = COLORS.get(e['kind'], (100, 100, 100))
            label = str(e['reading_order'])

            cv2.rectangle(img, (ix0, iy0), (ix1, iy1), color, 1)

            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.35, 1)
            cv2.rectangle(img, (ix0, max(iy0-th-4, 0)), (ix0+tw+4, max(iy0, th+4)), color, -1)
            cv2.putText(img, label, (ix0+2, max(iy0-3, th+1)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,255), 1, cv2.LINE_AA)

        out_path = os.path.join(out_folder, f'page_{page_num:03d}.png')
        cv2.imwrite(out_path, img)
        print(f'Saved → {out_path}')

    doc.close()
    print(f'Done. Output folder → {out_folder}')

In [7]:
# Save CSV
def save_csv(elements, out_path='elements.csv'):
    fields = ['reading_order','kind','page_num','pdf_x0','pdf_y0','pdf_x1','pdf_y1',
              'img_x0','img_y0','img_x1','img_y1','content']

    with open(out_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()

        for e in elements:
            x0,y0,x1,y1 = e['bbox_pdf']
            ix0,iy0,ix1,iy1 = e['bbox_img']

            writer.writerow({
                'reading_order': e['reading_order'],
                'kind': e['kind'],
                'page_num': e['page_num'],
                'pdf_x0': round(x0,2),
                'pdf_y0': round(y0,2),
                'pdf_x1': round(x1,2),
                'pdf_y1': round(y1,2),
                'img_x0': ix0,
                'img_y0': iy0,
                'img_x1': ix1,
                'img_y1': iy1,
                'content': e['content']
            })

    print(f'Saved → {out_path}')

In [8]:
# Run pipeline
elements = extract_elements(PDF)
elements = assign_reading_order(elements)
elements = attach_image_bboxes(elements, SCALE)

render_and_annotate(PDF, elements, SCALE, out_folder='annotated_pages_final')
save_csv(elements, out_path='elements.csv')

Saved → annotated_pages_final\page_001.png
Saved → annotated_pages_final\page_002.png
Saved → annotated_pages_final\page_003.png
Saved → annotated_pages_final\page_004.png
Saved → annotated_pages_final\page_005.png
Saved → annotated_pages_final\page_006.png
Saved → annotated_pages_final\page_007.png
Saved → annotated_pages_final\page_008.png
Saved → annotated_pages_final\page_009.png
Saved → annotated_pages_final\page_010.png
Saved → annotated_pages_final\page_011.png
Saved → annotated_pages_final\page_012.png
Saved → annotated_pages_final\page_013.png
Saved → annotated_pages_final\page_014.png
Saved → annotated_pages_final\page_015.png
Saved → annotated_pages_final\page_016.png
Saved → annotated_pages_final\page_017.png
Saved → annotated_pages_final\page_018.png
Done. Output folder → annotated_pages_final
Saved → elements.csv


In [10]:
# ── Preview CSV ───────────────────────────────────────────────
import pandas as pd
pd.set_option('display.max_colwidth', 60)
display(pd.read_csv('elements.csv').head(30))

,reading_order,kind,page_num,pdf_x0,pdf_y0,pdf_x1,pdf_y1,img_x0,img_y0,img_x1,img_y1,content
0,2,text,1,36.00,675.77,485.79,698.77,75,194,1012,242,General Dynamics Corporation NYSE:GD
1,3,text,1,36.00,640.42,456.89,665.42,75,263,951,315,FQ1 2025 Earnings Call Transcripts
2,4,text,1,36.00,614.92,380.03,632.92,75,331,791,368,"Wednesday, April 23, 2025 1:00 PM GMT"
3,5,text,1,36.00,593.07,332.02,609.07,75,381,691,414,S&P Global Market Intelligence Estimates
4,11,text,1,192.92,575.55,233.80,583.55,401,434,487,450,-FQ1 2025-
5,12,text,1,334.92,575.55,375.82,583.55,697,434,782,450,-FQ2 2025-
6,13,text,1,444.09,575.55,479.63,583.55,925,434,999,450,-FY 2025-
7,82,text,1,550.59,575.55,586.15,583.55,1147,434,1221,450,-FY 2026-
8,20,text,1,120.20,557.50,164.51,564.50,250,473,342,488,CONSENSUS
9,21,text,1,198.99,557.50,227.75,564.50,414,473,474,488,ACTUAL


In [18]:
def render_page_by_kind(pdf_path, csv_path, page_num, kind, scale=SCALE, out_folder='filtered_pages'):
    import pandas as pd
    os.makedirs(out_folder, exist_ok=True)

    df = pd.read_csv(csv_path)
    filtered = df[(df['page_num'] == page_num) & (df['kind'] == kind)]

    if filtered.empty:
        print(f'No elements of kind="{kind}" found on page {page_num}')
        return

    # ── render the real page ─────────────────────────────────────
    doc = fitz.open(pdf_path)
    pix = doc[page_num - 1].get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
    doc.close()

    real_page = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    real_page = cv2.cvtColor(real_page, cv2.COLOR_RGB2BGR)

    # ── blank white canvas same size ─────────────────────────────
    canvas = np.ones((pix.height, pix.width, 3), dtype=np.uint8) * 255

    # ── paste only the selected kind regions from real page ───────
    for _, row in filtered.iterrows():
        ix0, iy0, ix1, iy1 = int(row['img_x0']), int(row['img_y0']), int(row['img_x1']), int(row['img_y1'])
        canvas[iy0:iy1, ix0:ix1] = real_page[iy0:iy1, ix0:ix1]   # copy exact pixels

    out_path = os.path.join(out_folder, f'page_{page_num:03d}_{kind}.png')
    cv2.imwrite(out_path, canvas)
    print(f'Saved → {out_path}  ({len(filtered)} "{kind}" elements)')


# ── Call it ──────────────────────────────────────────────────────
# render_page_by_kind(PDF, 'elements.csv', page_num=1, kind='figure')
render_page_by_kind(PDF, 'elements.csv', page_num=1, kind='text')

Saved → filtered_pages\page_001_text.png  (64 "text" elements)
